In [ ]:
# Analizador léxico básico ya resuelto.

import re

def analizador_lexico(cadena):
    """
    Convierte una cadena de entrada en una lista de tokens.
    Retorna:
        [(TIPO_TOKEN, lexema), ...]
    """

    especificacion_tokens = [
        ('NUM',        r'\d+'),
        ('OP_REL',     r'>=|<=|==|!=|>|<'),
        ('PUNTO_COMA', r';'),
        ('PARENT_IZQ', r'\('),
        ('PARENT_DER', r'\)'),
        ('ID',         r'[A-Za-z_][A-Za-z0-9_]*'),
        ('ESPACIO',    r'[ \t\n]+'),
        ('ERROR',      r'.')
    ]

    patron = '|'.join(
        f'(?P<{nombre}>{expresion})'
        for nombre, expresion in especificacion_tokens
    )

    palabras_reservadas = {
        'WHEN',
        'THEN',
        'AND',
        'OR',
        'NOT',
        'TRUE',
        'FALSE'
    }

    tokens = []

    for coincidencia in re.finditer(patron, cadena):

        tipo = coincidencia.lastgroup
        lexema = coincidencia.group()

        if tipo == 'ESPACIO':
            continue

        elif tipo == 'ID' and lexema.upper() in palabras_reservadas:
            tipo = lexema.upper()

        elif tipo == 'ERROR':
            raise ValueError(
                f"Símbolo no reconocido: '{lexema}'"
            )

        tokens.append((tipo, lexema))

    return tokens

In [14]:
# Ejemplo de prueba.

entrada = "WHEN temperatura > 28 AND humedad < 40 THEN encender ventilador;"

tokens = analizador_lexico(entrada)

for token in tokens:
    print(token)

('WHEN', 'WHEN')
('ID', 'temperatura')
('OP_REL', '>')
('NUM', '28')
('AND', 'AND')
('ID', 'humedad')
('OP_REL', '<')
('NUM', '40')
('THEN', 'THEN')
('ID', 'encender')
('ID', 'ventilador')
('PUNTO_COMA', ';')


# Proyecto INFO1148: Analizador Sint?ctico LL(1) ? CyberGuard DSL
**Dominio:** Reglas Simples de Ciberseguridad y Control de Firewall (`CyberGuard DSL`)

Este notebook integra el analizador l?xico extendido, el c?lculo de conjuntos `FIRST` y `FOLLOW`, la construcci?n de la **Tabla Predictiva LL(1)**, el **Analizador Sint?ctico por Descenso Recursivo**, la generaci?n de **?rbol Sint?ctico / AST** y la evaluaci?n de **30 casos de prueba**.


In [2]:
import os, sys
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))
from main import mostrar_gramatica_y_tabla
mostrar_gramatica_y_tabla()


1. GRAMÁTICA LIBRE DE CONTEXTO (GLC) EN FORMA LL(1) - CYBERGUARD DSL
  (01) <programa>             -> <regla> <resto_programa>
  (02) <resto_programa>       -> <regla> <resto_programa> | ε
  (03) <regla>                -> WHEN <condicion> THEN <lista_acciones> <clausula_else> PUNTO_COMA
  (04) <condicion>            -> <termino_and> <resto_or>
  (05) <resto_or>             -> OR <termino_and> <resto_or> | ε
  (06) <termino_and>          -> <factor_not> <resto_and>
  (07) <resto_and>            -> AND <factor_not> <resto_and> | ε
  (08) <factor_not>           -> NOT <factor_not> | <primario>
  (09) <primario>             -> PARENT_IZQ <condicion> PARENT_DER | <comparacion>
  (10) <comparacion>          -> ID OP_REL <valor>
  (11) <valor>                -> NUM | TRUE | FALSE | ID
  (12) <lista_acciones>       -> <accion> <resto_acciones>
  (13) <resto_acciones>       -> COMA <accion> <resto_acciones> | ε
  (14) <accion>               -> ID <complemento_accion>
  (15) <complemento_accion>

In [3]:
from main import demostrar_analisis

sentencia = (
    'WHEN (intentos_ssh > 3 OR puerto_destino == 22) AND NOT ip_confiable == TRUE '
    'THEN bloquear_ip(ip_origen), alertar soc_admin ELSE registrar_acceso;'
)
demostrar_analisis(sentencia, guardar_archivos=False)



4. DEMOSTRACIÓN DE ANÁLISIS LÉXICO, SINTÁCTICO Y GENERACIÓN DE ÁRBOLES
Entrada analizada:
  WHEN (intentos_ssh > 3 OR puerto_destino == 22) AND NOT ip_confiable == TRUE THEN bloquear_ip(ip_origen), alertar soc_admin ELSE registrar_acceso;

Estado: Entrada sintácticamente válida (Aceptada).

Tokens reconocidos [(TIPO, lexema, línea, columna)]:
  (WHEN       , 'WHEN', lín=1, col=1)
  (PARENT_IZQ , '(', lín=1, col=6)
  (ID         , 'intentos_ssh', lín=1, col=7)
  (OP_REL     , '>', lín=1, col=20)
  (NUM        , '3', lín=1, col=22)
  (OR         , 'OR', lín=1, col=24)
  (ID         , 'puerto_destino', lín=1, col=27)
  (OP_REL     , '==', lín=1, col=42)
  (NUM        , '22', lín=1, col=45)
  (PARENT_DER , ')', lín=1, col=47)
  (AND        , 'AND', lín=1, col=49)
  (NOT        , 'NOT', lín=1, col=53)
  (ID         , 'ip_confiable', lín=1, col=57)
  (OP_REL     , '==', lín=1, col=70)
  (TRUE       , 'TRUE', lín=1, col=73)
  (THEN       , 'THEN', lín=1, col=78)
  (ID         , 'bloquear_ip'

In [4]:
from tests.test_suite import ejecutar_pruebas
ejecutar_pruebas(mostrar_tabla=True)


ID   | Categoría   | Esperado   | Obtenido   | Estado | Observación / Mensaje del Analizador
V01  | Válida      | Aceptada   | Aceptada   | OK     | Regla simple de fuerza bruta con operador '>' y acción verbo-objeto.
V02  | Válida      | Aceptada   | Aceptada   | OK     | Condición compuesta con AND, valor booleano FALSE y acción con parámetro numérico.
V03  | Válida      | Aceptada   | Aceptada   | OK     | Operador OR y múltiples acciones separadas por coma (extensión creativa).
V04  | Válida      | Aceptada   | Aceptada   | OK     | Uso del operador unario NOT y combinación de acción con objeto y acción simple.
V05  | Válida      | Aceptada   | Aceptada   | OK     | Agrupación con paréntesis, precedencia OR/AND y cláusula condicional ELSE.
V06  | Válida      | Aceptada   | Aceptada   | OK     | Tres acciones encadenadas con parámetros de tipo ID, NUM y booleano TRUE.
V07  | Válida      | Aceptada   | Aceptada   | OK     | Negación NOT aplicada sobre una agrupación entre paréntesis.